# Workshop 3W — Upsell Like an Engineer
Unit 3 — Programming for Intelligent Products

Today we’ll practise TDD (Red → Green → Refactor), experiment logging, and debugging discipline.

In [1]:
import sys, platform
print("Python:", sys.version)
print("Platform:", platform.platform())
import mlflow, pandas as pd
print("MLflow:", mlflow.__version__)
print("Environment ready ✅")
print(sys.executable)





Python: 3.14.2 (main, Sep 10 2026, 12:33:04) [GCC 13.3.0]
Platform: Linux-6.8.0-1064-azure-x86_64-with-glibc2.39
MLflow: 3.16.1
Environment ready ✅
/workspaces/ai6_workshop-3/.venv/bin/python


In [21]:
def should_upsell(order: dict) -> bool:
    time = str(order.get("time_of_day", "")).strip().lower()
    loyalty = str(order.get("loyalty_member", "no")).strip().lower()
    temp = int(order.get("temperature", 0) or 0)
    size = int(order.get("order_size", 0) or 0)

    if time == "lunch":
        return loyalty == "yes"

    if temp >= 30:
        return True

    if size >= 4:
        return True

    return False
        




# Positive test – loyalty at lunch → True
order_1a = {"time_of_day": "lunch", "loyalty_member": "yes"}
assert should_upsell(order_1a) is True, "FAIL: Loyalty at lunch should be upsold."
print("PASS: Test 1a")

order_1b = {"time_of_day": "lunch", "loyalty_member": "no"}
assert should_upsell(order_1b) is False, "FAIL: Non-loyalty at lunch should NOT be upsold."
print("PASS: Test 1b")

# Evening loyalty → should NOT be upsold
order_2 = {"time_of_day": "evening", "loyalty_member": "yes"}
assert should_upsell(order_2) is False, "FAIL: Evening loyalty should NOT be upsold."
print("PASS: Test 2")

# Heatwave (temperature 30 or greater) → True
order_3 = {"time_of_day": "evening", "loyalty_member": "no", "temperature": 32}
assert should_upsell(order_3) is True, "FAIL: Hot weather should trigger upsell."
print("PASS: Test 3")

# Large order (order size 4 or greater) → True
order_4 = {"order_size": 4}
assert should_upsell(order_4) is True, "FAIL: Large orders should trigger upsell."
print("PASS: Test 4")

# Busy lunch heatwave (non-loyalty) → should NOT upsell
order_5 = {"time_of_day": "lunch", "loyalty_member": "no", "temperature": 32}
assert should_upsell(order_5) is False, "FAIL: Busy lunch heatwave (non-loyalty) should NOT upsell."
print("PASS: Test 5")

PASS: Test 1a
PASS: Test 1b
PASS: Test 2
PASS: Test 3
PASS: Test 4
PASS: Test 5


In [20]:

import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("hotdog-upsell")

with mlflow.start_run(run_name="Initial Loyalty Rule"):
    order = {
        "time_of_day": "lunch",
        "loyalty_member": "yes"
    }

    prediction = should_upsell(order)

    mlflow.log_params(order)
    mlflow.log_metric("prediction", int(bool(prediction)))
    mlflow.set_tag("rule_version", "v1")

print("Experiment logged successfully ✅")


Experiment logged successfully ✅
